<a href="https://colab.research.google.com/github/RohanYashraj/ifoa-workshop/blob/main/notebooks_v2/01_genai_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · GenAI Basics — your first calls to the reasoner

**Agentic AI for Actuaries** · IFoA Workshop · 10 July 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Insurer is a fictional entity calibrated to plausible Indian market experience, for teaching only.

**Used in:** Session 1, Part 2 (The Reasoner).
**You will:** make your first Gemini API call, practise the CCCE prompt discipline, watch a hallucination happen on demand, and get guaranteed-parseable JSON out of an LLM.

**Setup (2 minutes):**
1. Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com) → *Get API key*.
2. In Colab, click the **key icon** (left sidebar) → *Add new secret* → name it `GOOGLE_API_KEY`, paste the key, toggle notebook access ON.
3. Run the cells top to bottom (`Runtime → Run all` after setup).

In [1]:
%pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.2/984.2 kB 26.0 MB/s eta 0:00:00


## §1 · Auth — the key never appears in the notebook
Colab Secrets keeps the key out of the notebook file. This is the same hygiene you will use for every agent you ship: secrets live in a store, never in code.

In [2]:
import os
from google import genai
from IPython.display import Markdown, display
from google.colab import userdata   # Colab-only; see comment below for local Jupyter

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# Local Jupyter alternative:
#   os.environ["GOOGLE_API_KEY"] = "..."  # or use python-dotenv

client = genai.Client()
MODEL = "gemini-3.1-flash-lite"   # PINNED — silent model drift is an audit failure
print("Client ready, model pinned to:", MODEL)

Client ready, model pinned to: gemini-3.1-flash-lite


## §2 · First call — define IBNR for a board member

In [3]:
response = client.models.generate_content(
    model=MODEL,
    contents="Define IBNR for a non-actuarial board member, in one line.",
)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n--- usage ---")
print(response.usage_metadata)   # token counts: you will care about these when agents multiply call volume

📋 GEMINI MODEL RESPONSE


IBNR represents the estimated cost of claims that have already occurred but have not yet been reported to or fully settled by the company.


END OF MODEL RESPONSE

--- usage ---
cache_tokens_details=None cached_content_token_count=None candidates_token_count=26 candidates_tokens_details=None prompt_token_count=19 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=19
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=45 traffic_type=None


## §3 · CCCE — Clarity, Context, Constraints, Examples
The prompt below is the worked example from the slides: an IBNR commentary for ABC Health Q3 2024. Each bracketed fragment does exactly one job — edit any part without breaking the others.

**Exercise:** delete the Constraints block, re-run, and compare. Then rewrite the prompt for *your* line of business.

In [4]:
ccce_prompt = """
[Clarity] Write a two-paragraph commentary on the IBNR result for ABC Health Q3 2024.
[Context] Indemnity health book. Chain-ladder ultimate INR 186 Cr vs prior estimate INR 172 Cr.
Q3 saw a hospital network strike in two states.
[Constraints] Audience: appointed actuary peer-review meeting. Max 180 words.
Do not invent figures. Cite only the figures provided above.
[Example] Voice to match: "The Q2 ultimate of INR 164 Cr increased to INR 172 Cr after the network
expansion in Tier 2 cities..."
"""
resp = client.models.generate_content(model=MODEL, contents=ccce_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


The Q3 2024 IBNR valuation for the indemnity health book reflects an ultimate estimate of INR 186 Cr, representing an increase from the prior estimate of INR 172 Cr. This upward development is primarily driven by recent disruptions within our hospital network, specifically the strike action observed across two states during the quarter. The resulting shift in utilization patterns has led to a deceleration in claim reporting, necessitating a recalibration of our development factors to capture the latent cost emergence.

While the chain-ladder methodology continues to underpin our projections, the volatility introduced by the strike required additional actuarial judgment to avoid understating the ultimate liability. We have monitored the reporting lags closely to ensure the revised estimate of INR 186 Cr adequately reflects the anticipated claims backlog as services normalize. Further sensitivity testing will be performed in Q4 to confirm the stability of these assumptions as network access stabilizes and the remaining claims crystallize.


END OF MODEL RESPONSE


### §3.1 · Demo 1 — the vague version, for contrast
Run the deliberately vague prompt below, then re-run the CCCE version above and **diff the outputs**. Same model, same cost — the entire quality delta is the prompt.

In [5]:
vague = "Write about IBNR for our board."
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=vague).text[:800]))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Expect: a generic essay that INVENTS plausible numbers (we gave it none)
# and lands in a register somewhere between textbook and LinkedIn.

📋 GEMINI MODEL RESPONSE


This briefing is designed for a Board of Directors level, focusing on the strategic and financial implications of **IBNR (Incurred But Not Reported)** rather than the granular actuarial mechanics.

***

### Executive Briefing: Understanding IBNR
**Subject:** Risk Management and Financial Oversight of IBNR Reserves

#### 1. What is IBNR?
In the simplest terms, **IBNR** stands for **Incurred But Not Reported**. It is an accounting estimate representing the liability for claims that have already occurred (the event has happened) but have not yet been reported to the company as of the financial reporting date.

*   **The Concept:** If a policyholder suffers a loss on December 28th, but the claim is not filed or processed until January 5th, the cost of that claim belongs to the previous year’s 


END OF MODEL RESPONSE


### §3.2 · Demo 2 — one fact, two audiences
Audience is a prompt parameter. Same reserve-strengthening fact, rendered for a board member and for a new student. Note: the model *dresses* the fact we supply — it does not source it.

In [6]:
fact = ("We strengthened motor BI reserves by INR 42 Cr "
        "following the new tribunal award benchmarks.")

for audience, style in [
    ("board member", "2 sentences, business impact first, no jargon"),
    ("new actuarial student", "4 sentences, explain WHY tribunal awards drive BI reserves, define terms"),
]:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Explain: {fact} For a {audience}. {style}")
    # Clear visual separation
    print("=" * 70)
    print("📋 GEMINI MODEL RESPONSE")
    print("=" * 70)

    display(Markdown(f"**{audience.upper()}**\n\n{r.text}"))

    print("\n" + "=" * 70)
    print("END OF MODEL RESPONSE")
    print("=" * 70)

📋 GEMINI MODEL RESPONSE


**BOARD MEMBER**

We increased our financial reserves by INR 42 crore to ensure we remain fully protected against recent increases in court-mandated accident compensation payouts. This proactive adjustment strengthens our balance sheet and prevents future earnings volatility related to these rising legal obligations.


END OF MODEL RESPONSE
📋 GEMINI MODEL RESPONSE


**NEW ACTUARIAL STUDENT**

In motor insurance, **Bodily Injury (BI) reserves** are the funds set aside to cover potential future compensation payouts for claimants injured in accidents. **Tribunal awards** act as the legal benchmarks that dictate how much compensation an insurer must pay for specific injuries, effectively setting the "price" of these claims. When judicial rulings or government notifications increase these benchmark awards, your existing liability estimates become insufficient to cover the higher projected costs. Consequently, you must **strengthen reserves**—adding the extra INR 42 Cr—to ensure the company’s balance sheet accurately reflects the increased legal liability for these outstanding claims.


END OF MODEL RESPONSE


### §3.3 · Demo 3 — few-shot examples tame formatting
Show, don't tell: two worked examples buy you the delimiter, the casing, the arrow convention, and no chatty preamble. **Exercise:** feed it a genuinely weird input and see whether the pattern holds.

In [7]:
prompt = """Convert each change to the format of the examples.

EXAMPLES
In: We moved lapse from 6% to 5.5% for durations 2+.
Out: LAPSE | dur 2+ | 6.0% -> 5.5%
In: Expense inflation up 50bps.
Out: EXPENSE_INFL | all | +50bps

NOW CONVERT
In: Mortality improvement for males 45-60 moves from 1.5% to 1.25%.
Out:"""
print(client.models.generate_content(model=MODEL, contents=prompt).text)

Out: MORT_IMP | male 45-60 | 1.5% -> 1.25%


### §3.4 · Demo 4 — step-by-step reasoning (with a warning label)
Asking for steps improves reliability — it does **not** guarantee it. Re-run this cell three times: do the running totals stay identical? This is why the afternoon's agent does arithmetic in *Python* and lets Gemini narrate.

In [8]:
prompt = """A motor policy has base premium INR 6,500 with relativities:
vehicle age 6-9yrs = 1.15, SUV = 1.20, Tier2 = 1.00, NCB 35% = 0.65.
Walk through the premium calculation STEP BY STEP, showing the running
total after each factor, then state the final premium."""
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=prompt).text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Check by hand: 6500 * 1.15 * 1.20 * 1.00 * 0.65 = 5,830.50

📋 GEMINI MODEL RESPONSE


To calculate the final motor premium, we apply each relativity factor sequentially to the base premium. 

Here is the step-by-step breakdown:

### Step 1: Base Premium
*   **Starting Amount:** INR 6,500

### Step 2: Apply Vehicle Age Factor (1.15)
*   6,500 × 1.15 = 7,475
*   **Running Total:** **INR 7,475**

### Step 3: Apply Vehicle Type Factor (1.20)
*   7,475 × 1.20 = 8,970
*   **Running Total:** **INR 8,970**

### Step 4: Apply Tier Factor (1.00)
*   8,970 × 1.00 = 8,970
*   **Running Total:** **INR 8,970**

### Step 5: Apply NCB Factor (0.65)
*   8,970 × 0.65 = 5,830.50
*   **Running Total:** **INR 5,830.50**

***

### Final Premium
The final premium for the motor policy is **INR 5,830.50**.


END OF MODEL RESPONSE


### §3.5 · Demo review — the habit that IS the skill
1. CCCE moved quality more than a model upgrade would — specification beats horsepower.
2. Register control is leverage, but the model dresses facts; it doesn't source them.
3. Few-shot is a formatting contract — stress-test it before relying on it.
4. Step-by-step is transparency, not verified arithmetic.

**The loop:** prompt → output → review → edit prompt — the same loop you'll run on agent traces this afternoon.

## §4 · The hallucination demo — run it, believe it
We ask for a regulation that **does not exist**. The model will not say 'no such factor' — it will produce the most *plausible-sounding* answer, confidently.

⚠️ This exact failure mode reappears **inside your agent** in notebook 04 — and you will fix it with a guardrail tool.

In [9]:
hallucination_prompt = (
    "What is the IRDAI motor tariff factor for hatchbacks under 1000cc? "
    "Give the exact factor value and the section reference."
)
resp = client.models.generate_content(model=MODEL, contents=hallucination_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n⚠️  Verify before you trust: there is no such published factor. "
      "Whatever appears above was constructed to be plausible, not true.")

📋 GEMINI MODEL RESPONSE


The motor insurance premiums in India are determined by the **India Motor Tariff (IMT)**, specifically under the **All India Motor Tariff (AIMT)** guidelines.

For private cars (including hatchbacks) with an engine capacity **not exceeding 1000cc**, the tariff details are as follows:

### 1. The Exact Factor Value
The "factor" you are referring to is the **Basic Premium rate** set by the IRDAI. As per the current **Long-Term/Annual Third-Party (TP) Liability Premium rates** (last notified via the Gazette of India on June 1, 2022), the annual premium for cars not exceeding 1000cc is:

*   **₹2,094**

*(Note: This is the mandatory Third-Party base premium. While not a percentage "factor," this is the fixed statutory value used for all pricing calculations.)*

### 2. Section Reference
The pricing for motor vehicles is governed by **Section 1 (Private Cars)** of the **Indian Motor Tariff**.

*   **Specific Reference:** You will find this under **Section 1, Sub-section A (Third Party Premium)** of the **India Motor Tariff (AIMT)**.
*   **Latest Notification:** The specific rates currently in effect are governed by the **S.O. 2486(E) notification** issued by the Ministry of Road Transport and Highways (MoRTH) in consultation with the IRDAI, dated **June 1, 2022**.

### Important Context:
*   **The "Factor" vs. "OD Premium":** While the Third-Party premium is fixed by the IRDAI/Government (the ₹2,094 figure), the **Own Damage (OD)** portion of the premium is **de-tariffed**. This means there is no fixed IRDAI "factor" for OD premiums; insurance companies calculate this based on their own actuarial risk models, vehicle age, IDV (Insured Declared Value), and geographical zone.
*   **Hatchbacks vs. Sedans:** The tariff is strictly classified by engine capacity (cc), not body style. Any hatchback under 1000cc falls into the same category as a sedan under 1000cc for the purpose of Third-Party pricing.

**Where to view the official documentation:**
You can verify these figures by accessing the [IRDAI website](https://www.irdai.gov.in/) under the "Motor Insurance" regulatory section or the [Ministry of Road Transport and Highways (MoRTH)](https://morth.nic.in/) gazette notifications.


END OF MODEL RESPONSE

⚠️  Verify before you trust: there is no such published factor. Whatever appears above was constructed to be plausible, not true.


## §5 · Structured output — because agents speak JSON
One config line guarantees parseable JSON. This is how every component of an agentic system exchanges data — prose is only for humans at the last step.

In [10]:
import json

prompt = """For private car comprehensive insurance, list 5 rating factors.
For each: name, direction (increase/decrease premium), one-line justification. Return JSON."""

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"response_mime_type": "application/json"},
)
factors = json.loads(resp.text)   # guaranteed to parse
for f in factors:
    print(f)

{'name': 'Driver Age', 'direction': 'decrease', 'justification': 'Younger, inexperienced drivers are statistically more likely to be involved in accidents than older, experienced drivers.'}
{'name': 'Vehicle Value', 'direction': 'increase', 'justification': 'Higher-valued vehicles cost more to repair or replace in the event of a total loss or significant damage.'}
{'name': 'Claims History', 'direction': 'increase', 'justification': 'A history of previous claims indicates a higher statistical probability of filing future claims.'}
{'name': 'Annual Mileage', 'direction': 'increase', 'justification': 'Spending more time on the road increases the duration of exposure to potential traffic accidents.'}
{'name': 'Garaging Location', 'direction': 'increase', 'justification': 'Areas with high rates of crime, vandalism, or traffic congestion carry higher risks for insurers.'}


## §6 · Review exercise — mark the model's homework
Treat the JSON above as a junior analyst's first draft and grade it:

1. Is every **direction** consistent with your priors?
2. Did it name factors your book doesn't collect (e.g. telematics, annual mileage)?
3. What material factors are **missing** (vehicle make? segment?)
4. What would you still need before any of this goes near a tariff filing? *(Hint: magnitudes → a GLM run → notebook 02.)*

**The rule that survives today:** the reasoner narrates; tools know; humans sign.

---
**Log what you ran.** For anything regulatory: save the full prompt–response pair, the model id, and the timestamp — 'the AI wrote it' is not a defence without the receipt.

In [11]:
# Minimal call log — one CSV row per call. In production this is your observability stack.
import datetime, csv, pathlib

def log_call(prompt, response_text, model=MODEL, path="genai_call_log.csv"):
    new = not pathlib.Path(path).exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["ts_utc", "model", "prompt", "response"])
        w.writerow([datetime.datetime.utcnow().isoformat(), model, prompt, response_text])

log_call(prompt, resp.text)
print("logged — this habit is checklist question 10 in miniature")

logged — this habit is checklist question 10 in miniature


/tmp/ipykernel_2178/3299675427.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  w.writerow([datetime.datetime.utcnow().isoformat(), model, prompt, response_text])
